In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score, f1_score

In [2]:
df = pd.read_csv("prehackathonsup/train_data/train_data.csv")
df_test = pd.read_csv("prehackathonsup/test_data/test_data.csv")

In [3]:
df

,engine_no,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,RUL
0,0,1,25.0074,0.6200,60.0,462.54,536.84,1256.52,1043.97,7.05,...,84.93,14.35,8.4712,NaN,NaN,NaN,NaN,NaN,NaN,339
1,0,2,35.0072,0.8413,100.0,449.44,555.44,1364.42,1128.75,5.48,...,100.00,14.88,8.9928,NaN,NaN,NaN,NaN,NaN,NaN,338
2,0,3,25.0053,0.6215,60.0,462.54,536.42,1265.94,1047.23,7.05,...,84.93,14.21,8.5107,NaN,NaN,NaN,NaN,NaN,NaN,337
3,0,4,42.0045,0.8407,100.0,445.00,549.41,1355.52,1115.81,3.91,...,100.00,10.63,6.4578,NaN,NaN,NaN,NaN,NaN,NaN,336
4,0,5,35.0046,0.8400,100.0,449.44,555.21,1361.04,1123.63,5.48,...,100.00,14.95,9.0279,NaN,NaN,NaN,NaN,NaN,NaN,335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160354,708,159,10.0040,0.2519,100.0,489.05,605.81,1508.72,1333.13,10.52,...,100.00,28.48,16.8884,NaN,NaN,NaN,NaN,NaN,NaN,4
160355,708,160,10.0074,0.2500,100.0,489.05,605.83,1509.90,1328.53,10.52,...,100.00,28.20,16.9498,NaN,NaN,NaN,NaN,NaN,NaN,3
160356,708,161,34.9982,0.8400,100.0,449.44,556.62,1374.56,1145.17,5.48,...,100.00,14.76,8.9228,NaN,NaN,NaN,NaN,NaN,NaN,2
160357,708,162,24.9993,0.6219,60.0,462.54,537.58,1274.92,1064.82,7.05,...,84.93,14.05,8.3890,NaN,NaN,NaN,NaN,NaN,NaN,1


In [4]:
print(df["time_in_cycles"].mean())
print(df.groupby('engine_no')['time_in_cycles'].max().mean())

123.33133781078705
226.17630465444287


In [5]:
df_test

,engine_no,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27
0,0,1,42.0034,0.8400,100.0,445.00,549.36,1342.05,1124.56,3.91,...,2212,100.0,10.69,6.3956,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,42.0017,0.8400,100.0,445.00,548.83,1351.93,1116.28,3.91,...,2212,100.0,10.55,6.3775,NaN,NaN,NaN,NaN,NaN,NaN
2,0,3,0.0028,0.0019,100.0,518.67,642.35,1583.74,1400.44,14.62,...,2388,100.0,38.85,23.3483,NaN,NaN,NaN,NaN,NaN,NaN
3,0,4,42.0047,0.8400,100.0,445.00,549.69,1354.36,1125.55,3.91,...,2212,100.0,10.56,6.4871,NaN,NaN,NaN,NaN,NaN,NaN
4,0,5,10.0058,0.2506,100.0,489.05,604.72,1496.65,1310.52,10.52,...,2319,100.0,28.78,17.1987,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104892,706,115,-0.0022,0.0002,100.0,518.67,642.69,1595.77,1413.75,14.62,...,2388,100.0,38.90,23.3045,NaN,NaN,NaN,NaN,NaN,NaN
104893,706,116,0.0018,-0.0001,100.0,518.67,643.26,1590.79,1407.73,14.62,...,2388,100.0,38.95,23.2379,NaN,NaN,NaN,NaN,NaN,NaN
104894,706,117,-0.0047,-0.0004,100.0,518.67,642.78,1590.92,1410.99,14.62,...,2388,100.0,38.63,23.2412,NaN,NaN,NaN,NaN,NaN,NaN
104895,706,118,-0.0008,0.0001,100.0,518.67,642.85,1588.09,1413.42,14.62,...,2388,100.0,38.75,23.3305,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print(df_test["time_in_cycles"].mean())
print(df_test.groupby('engine_no')['time_in_cycles'].max().mean())

95.40658932095293
148.36916548797737


In [7]:
THRESHOLD = 100
df['label'] = (df['RUL'] <= THRESHOLD).astype(int)

In [8]:
df = df.drop(columns=[f"sensor_{i}" for i in range(22, 28)])
df_test = df_test.drop(columns=[f"sensor_{i}" for i in range(22, 28)])

constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
df = df.drop(columns=constant_cols)
df_test = df_test.drop(columns=constant_cols)

In [9]:
sensor_cols = [
    col for col in df.columns
    if col.startswith("sensor_")
]

trend (degradation signal)

In [10]:
diff_features = (
    df.groupby('engine_no')[sensor_cols]
      .diff()
      .fillna(0)
)

diff_features.columns = [f"{col}_diff" for col in sensor_cols]

In [11]:
df = pd.concat(
    [df, diff_features],
    axis=1
)

In [12]:
diff_features_test = (df_test.groupby('engine_no')[sensor_cols].diff().fillna(0))
diff_features_test.columns = [f"{col}_diff" for col in sensor_cols]
df_test = pd.concat([df_test, diff_features_test], axis=1)

In [13]:
sensor_cols += diff_features.columns.to_list()

In [14]:
window = 5

rolling_mean = (
    df.groupby('engine_no')[sensor_cols]
      .rolling(window, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

rolling_mean.columns = [f"{col}_mean" for col in sensor_cols]

rolling_std = (
    df.groupby('engine_no')[sensor_cols]
      .rolling(window, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
)

rolling_std.columns = [f"{col}_std" for col in sensor_cols]

In [15]:
df = pd.concat(
    [df, rolling_mean, rolling_std],
    axis=1
)

In [16]:
df

,engine_no,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12_diff_std,sensor_13_diff_std,sensor_14_diff_std,sensor_15_diff_std,sensor_16_diff_std,sensor_17_diff_std,sensor_18_diff_std,sensor_19_diff_std,sensor_20_diff_std,sensor_21_diff_std
0,0,1,25.0074,0.6200,60.0,462.54,536.84,1256.52,1043.97,7.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,35.0072,0.8413,100.0,449.44,555.44,1364.42,1128.75,5.48,...,13.378460,254.431162,139.759655,1.154210,0.000000,19.798990,217.788889,10.656099,0.374767,0.368827
2,0,3,25.0053,0.6215,60.0,462.54,536.42,1265.94,1047.23,7.05,...,19.050148,359.810000,192.964121,1.642511,0.000000,27.501515,308.000000,15.070000,0.601360,0.501980
3,0,4,42.0045,0.8407,100.0,445.00,549.41,1355.52,1115.81,3.91,...,22.943914,344.469220,187.979194,1.553655,0.000000,25.382080,292.052935,14.428427,1.833630,1.111372
4,0,5,35.0046,0.8400,100.0,449.44,555.21,1361.04,1123.63,5.48,...,33.702356,301.014743,165.718739,1.355679,0.000000,22.029526,254.502063,12.608467,2.834458,1.677969
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160354,708,159,10.0040,0.2519,100.0,489.05,605.81,1508.72,1333.13,10.52,...,230.417689,1.037916,70.655344,0.748360,0.007071,39.130551,105.321888,0.000000,16.224818,9.686379
160355,708,160,10.0074,0.2500,100.0,489.05,605.83,1509.90,1328.53,10.52,...,220.490352,1.033513,70.655028,0.738764,0.007071,37.848382,100.700050,0.000000,15.550670,9.278317
160356,708,161,34.9982,0.8400,100.0,449.44,556.62,1374.56,1145.17,5.48,...,236.028736,1.220545,69.319110,0.764332,0.008367,41.052406,109.216299,0.000000,16.695865,9.958633
160357,708,162,24.9993,0.6219,60.0,462.54,537.58,1274.92,1064.82,7.05,...,199.219544,161.157317,100.047841,0.893522,0.008367,37.407219,154.737843,6.739509,14.313271,8.517086


In [17]:
len(df.columns)

133

In [18]:
rolling_mean_test = (df_test.groupby('engine_no')[sensor_cols].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True))
rolling_mean_test.columns = [f"{col}_mean" for col in sensor_cols]
rolling_std_test = (df_test.groupby('engine_no')[sensor_cols].rolling(window, min_periods=1).std().reset_index(level=0, drop=True))
rolling_std_test.columns = [f"{col}_std" for col in sensor_cols]
df_test = pd.concat([df_test, rolling_mean_test, rolling_std_test], axis=1)

In [19]:
groups = df['engine_no']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

In [20]:
mean_cols = [col for col in df.columns if col.endswith("_mean")]
diff_cols = [col for col in df.columns if col.endswith("_diff")]

feature_cols = (
    ['op_setting_1', 'op_setting_2', 'op_setting_3']
    + mean_cols
    + diff_cols
)

In [21]:
X_train = train_df[feature_cols]
y_train = train_df['RUL']
y_train_classification = train_df['label']

X_test = test_df[feature_cols]
y_test = test_df['RUL']
y_test_classification = test_df['label']

In [22]:
X_train

,op_setting_1,op_setting_2,op_setting_3,sensor_1_mean,sensor_2_mean,sensor_3_mean,sensor_4_mean,sensor_5_mean,sensor_6_mean,sensor_7_mean,...,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_16_diff,sensor_17_diff,sensor_18_diff,sensor_19_diff,sensor_20_diff,sensor_21_diff
0,25.0074,0.6200,60.0,462.540000,536.8400,1256.520000,1043.970000,7.050000,9.020000,175.290000,...,0.00,0.00,0.00,0.0000,0.00,0.0,0.0,0.00,0.00,0.0000
1,35.0072,0.8413,100.0,455.990000,546.1400,1310.470000,1086.360000,6.265000,8.510000,185.000000,...,18.92,359.82,197.65,-1.6323,0.00,28.0,308.0,15.07,0.53,0.5216
2,25.0053,0.6215,60.0,458.173333,542.9000,1295.626667,1073.316667,6.526667,8.683333,181.763333,...,-19.18,-359.80,-188.24,1.6527,0.00,-27.0,-308.0,-15.07,-0.67,-0.4821
3,42.0045,0.8407,100.0,454.880000,544.5275,1310.600000,1083.940000,5.872500,7.942500,171.052500,...,-33.82,359.73,208.20,-1.5620,0.00,24.0,297.0,15.07,-3.58,-2.0529
4,35.0046,0.8400,100.0,453.792000,546.6640,1320.688000,1091.878000,5.794000,7.954000,175.824000,...,52.35,0.06,-14.90,-0.0147,0.00,3.0,11.0,0.00,4.32,2.5701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160354,10.0040,0.2519,100.0,487.480000,603.9700,1501.052000,1300.372000,10.098000,14.856000,373.404000,...,187.17,-1.19,67.91,-0.6883,0.01,38.0,96.0,0.00,13.82,8.1039
160355,10.0074,0.2500,100.0,487.480000,603.9780,1501.552000,1301.924000,10.098000,14.856000,373.224000,...,0.04,-0.01,1.56,-0.0071,0.00,-1.0,0.0,0.00,-0.28,0.0614
160356,34.9982,0.8400,100.0,479.130000,593.6500,1476.272000,1277.970000,9.324000,13.726000,345.172000,...,-187.25,1.44,-63.43,0.6785,-0.01,-35.0,-96.0,0.00,-13.44,-8.0270
160357,24.9993,0.6219,60.0,467.904000,572.4660,1409.904000,1202.780000,7.810000,11.210000,269.848000,...,-19.29,-359.91,-201.63,1.6614,0.00,-29.0,-308.0,-15.07,-0.71,-0.5338


In [23]:
len(X_train.columns)

66

In [24]:
len(X_test.columns)

66

In [25]:
model = HistGradientBoostingRegressor(
    random_state=42,
)

In [26]:
model.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide ` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 

In [27]:
y_pred = model.predict(X_test)

In [28]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

RMSE: 54.2162
R2 Score: 0.5557


In [29]:
model = HistGradientBoostingClassifier(
    random_state=42,
)

In [30]:
model.fit(X_train, y_train_classification)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [31]:
proba = model.predict_proba(X_test)[:, 1]

best_f1 = 0
best_t = 0.5

for t in np.linspace(0.1, 0.9, 50):
    preds = (proba >= t).astype(int)
    f1 = f1_score(y_test_classification, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"Best threshold: {best_t:.3f}")
print(f"Best F1: {best_f1:.4f}")

Best threshold: 0.443
Best F1: 0.8124


In [32]:
print(proba[:100])

[0.11078107 0.06247169 0.15499242 0.15142072 0.31965716 0.41667881
 0.44725799 0.19121733 0.20113005 0.28415038 0.29024861 0.1496432
 0.21481425 0.31525907 0.14914811 0.21556576 0.30928963 0.31846877
 0.22737068 0.25200372 0.16574922 0.1273537  0.2980918  0.30236644
 0.32415237 0.24936021 0.4153057  0.50716719 0.29917655 0.27459273
 0.24433589 0.28738197 0.34754731 0.32858705 0.20253755 0.26677027
 0.32588073 0.258291   0.17303585 0.25325377 0.17849961 0.28013436
 0.28159119 0.20100065 0.35583423 0.25113221 0.50699289 0.49353614
 0.31841062 0.2197254  0.33589045 0.30360868 0.33350227 0.37522951
 0.40899174 0.40806485 0.33477143 0.24897019 0.30596485 0.19930997
 0.18285158 0.27006279 0.19458409 0.22781905 0.32646383 0.40485742
 0.42639824 0.34616369 0.37675106 0.35533257 0.2690607  0.47275263
 0.39090222 0.66025487 0.63837667 0.15428061 0.52834923 0.43801649
 0.39099872 0.21444133 0.44051027 0.58545517 0.65954165 0.77252629
 0.65553099 0.63171342 0.51781922 0.67474937 0.46361848 0.43541

In [33]:
print((pd.Series(proba >= best_t).astype(int).value_counts()))

0    18812
1    13030
Name: count, dtype: int64


In [34]:
model = HistGradientBoostingClassifier(
    random_state=42,
)

In [35]:
model.fit(df[feature_cols], df['label'])

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [36]:
submit_proba = model.predict_proba(df_test[feature_cols])[:, 1]

submission = pd.DataFrame({
    'engineno': df_test['engine_no'].values,
    'result': (submit_proba >= best_t).astype(int)
})

Take last cycle per engine

In [37]:
# last_rows = df.groupby('engine_no').tail(1)
# submit_proba = model.predict_proba(last_rows[feature_cols])[:, 1]

# submission = pd.DataFrame({
#     'engineno': last_rows['engine_no'].values,
#     'result': (submit_proba >= best_t).astype(int)
# })

In [38]:
submit_proba[:100]

array([0.02203341, 0.0287815 , 0.00785755, 0.13705361, 0.11276801,
       0.17325804, 0.11858093, 0.12694016, 0.13513571, 0.12569162,
       0.12360716, 0.12141029, 0.16779938, 0.26567299, 0.10159119,
       0.14389584, 0.13307077, 0.09677564, 0.11440807, 0.14635791,
       0.14571247, 0.12226077, 0.11633061, 0.09439506, 0.11555733,
       0.09570928, 0.10239791, 0.18713703, 0.23780141, 0.19993471,
       0.19315822, 0.12678044, 0.11675089, 0.10207018, 0.1577124 ,
       0.19555705, 0.08050776, 0.13618396, 0.09460521, 0.10215093,
       0.09970844, 0.12959922, 0.07474096, 0.10109114, 0.1864053 ,
       0.11246473, 0.12640397, 0.16492683, 0.12908838, 0.15716834,
       0.12378838, 0.10494615, 0.10032453, 0.10751639, 0.23921222,
       0.14335816, 0.09868258, 0.14103723, 0.10816701, 0.06688657,
       0.064784  , 0.06295256, 0.10367164, 0.10925034, 0.12037657,
       0.11449494, 0.1970298 , 0.15696111, 0.14041125, 0.14714778,
       0.21758802, 0.29785241, 0.21886394, 0.19529601, 0.14013

In [39]:
submission.to_csv("submission.csv", index=False)

print("submission.csv created")
print(submission['result'].value_counts())

submission.csv created
result
0    82427
1    22470
Name: count, dtype: int64
